# ODI to Databricks Migration: TRG_EMP Load**Source File:** TARGET_FILE.sql**Conversion Date:** 2024-07-30This notebook performs a direct load into the `TRG_EMP` target table from the `EMPLOYEES` source table.

In [ ]:
# COMMAND ----------# DBTITLE 1,Create ETL Parameter Widgetsdbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "Datasource Number ID")dbutils.widgets.text("ETL_PROC_WID", "0", "ETL Process WID")dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

# ETL ParametersThese parameters capture contextual information for the ETL run, typically populated by an orchestrator.

In [ ]:
%sql
-- DBTITLE 1,Create Temporary Views for ETL ParametersCREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS SELECT '${ETL_JOB_TYPE}' AS etl_job_type;CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS SELECT CAST('${ODI_SESS_NO}' AS STRING) AS odi_sess_no;

In [ ]:
# COMMAND ----------# DBTITLE 1,Display ETL Parameter Valuesdisplay(spark.sql("SELECT * FROM v_etl_job_type"))display(spark.sql("SELECT * FROM v_datasource_num_id"))display(spark.sql("SELECT * FROM v_etl_proc_wid"))display(spark.sql("SELECT * FROM v_odi_sess_no"))

# Direct Target LoadThis section performs a direct insert of data from the source table into the target table.

In [ ]:
%sql
-- DBTITLE 1,Insert into Target Table (SCEN_TASK_NO {30})INSERT  INTO workspace.hr.trg_emp  (    employee_id ,    first_name ,    last_name ,    email ,    phone_number ,    hire_date ,    job_id ,    salary ,    commission_pct ,    manager_id ,    department_id  )SELECT  employees.employee_id ,  employees.first_name ,  employees.last_name ,  employees.email ,  employees.phone_number ,  employees.hire_date ,  employees.job_id ,  employees.salary ,  employees.commission_pct ,  employees.manager_id ,  employees.department_idFROM  workspace.hr.employees AS employees;

In [ ]:
%sql
-- DBTITLE 1,Record Count After InsertSELECT COUNT(*) AS trg_emp_record_count FROM workspace.hr.trg_emp;

# Optimize TargetThis step optimizes the target Delta table for query performance.

In [ ]:
%sql
-- DBTITLE 1,Optimize Target Table with ZORDER-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATSSET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;OPTIMIZE workspace.hr.trg_emp ZORDER BY (employee_id);

# CleanupThis section handles the cleanup of temporary tables, if any were created.

# ValidationThis section provides validation queries to ensure data integrity and correctness.

In [ ]:
%sql
-- DBTITLE 1,Final Record Count in Target TableSELECT COUNT(*) AS final_trg_emp_record_count FROM workspace.hr.trg_emp;

In [ ]:
%sql
-- DBTITLE 1,Sample Records from Target TableSELECT * FROM workspace.hr.trg_emp LIMIT 100;

# Conversion Notes and Manual Actions Required1.  **Schema and Table Names:** All schema and table names have been converted to `workspace.<schema_lower>.<table_lower>` format. Ensure the `hr` schema exists in your Databricks workspace.2.  **Oracle Hints Removed:** Oracle-specific hints like `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Delta Lake.3.  **Data Types:** The `INSERT...SELECT` statement assumes that the `workspace.hr.trg_emp` table already exists with compatible data types. If `TRG_EMP` needs to be created, ensure Oracle data types (e.g., `NUMBER`, `VARCHAR2`, `DATE`) are mapped correctly to Spark SQL types (`BIGINT`, `STRING`, `TIMESTAMP`) as per Section 5 of the conversion rules.4.  **ZORDER BY Column:** `employee_id` was chosen as a ZORDER BY column based on common practices for primary keys. Review if this is the most effective column(s) for your query patterns and adjust if necessary.